# So sánh kiến trúc: MobileNetV3Large, EfficientNet-B0, ViT-B/16 trên PlantVillage

Notebook này chạy `compare_models.py` của dự án **Bác sĩ cây trồng** trên GPU miễn phí của Colab. Ba kiến trúc dùng **cùng dữ liệu, cùng cách chia 70/15/15 (seed 42, giống `train.py`), cùng augmentation, cùng quy trình 2 giai đoạn** (học lớp phân loại với LR 1e-3, rồi tinh chỉnh một phần backbone với LR 1e-5) và được đánh giá trên **cùng tập test**.

Kết quả mỗi kiến trúc: accuracy, top-3, macro-F1, ma trận nhầm lẫn, số tham số, dung lượng file, thời gian suy luận CPU (ms/ảnh) và thời gian huấn luyện. Bảng tổng hợp `summary.csv` + biểu đồ `summary.png` hiện trong trang quản trị → *Đánh giá mô hình* → *So sánh kiến trúc* sau khi chép thư mục `results/compare/` về máy.

**Trước khi chạy:** Runtime → Change runtime type → **GPU (T4)**. Toàn bộ 54.000 ảnh với 8 + 4 epoch mất khoảng 1,5–2,5 giờ trên T4 (ViT chậm nhất). Thử nhanh bằng `LIMIT_PER_CLASS` ở ô cấu hình.

In [ ]:
#@title 1. Kiểm tra GPU và cài gói
!nvidia-smi -L || echo "Chưa bật GPU: Runtime -> Change runtime type -> GPU"
%pip install -q keras-hub scikit-learn pandas matplotlib
import tensorflow as tf, keras, keras_hub
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| keras-hub", keras_hub.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
#@title 2. Lấy compare_models.py (tải lên từ máy, hoặc chép từ Google Drive)
import os
SCRIPT_IN_DRIVE = ""  #@param {type:"string"}
if SCRIPT_IN_DRIVE:  # VD: /content/drive/MyDrive/phathiensaubenh/compare_models.py
    from google.colab import drive; drive.mount("/content/drive")
    !cp "$SCRIPT_IN_DRIVE" /content/compare_models.py
elif not os.path.exists("/content/compare_models.py"):
    from google.colab import files
    print("Chọn file compare_models.py trong thư mục dự án (D:\\phathiensaubenh)")
    up = files.upload()
    assert "compare_models.py" in up, "Cần tải đúng file compare_models.py"
print("OK:", os.path.getsize("/content/compare_models.py"), "byte")

In [ ]:
#@title 3. Dữ liệu PlantVillage (zip trên Google Drive hoặc Kaggle)
#@markdown Cách A: chép file zip `Plant_leave_diseases_dataset_without_augmentation.zip` lên Google Drive rồi điền đường dẫn.
#@markdown Cách B: để trống và dùng Kaggle (cần tải `kaggle.json` lên): dataset `abdallahalidev/plantvillage-dataset`.
ZIP_IN_DRIVE = "/content/drive/MyDrive/phathiensaubenh/Plant_leave_diseases_dataset_without_augmentation.zip"  #@param {type:"string"}
DATA_DIR = "/content/dataset/Plant_leave_diseases_dataset_without_augmentation"
import os, glob
if not os.path.isdir(DATA_DIR):
    os.makedirs("/content/dataset", exist_ok=True)
    if ZIP_IN_DRIVE:
        from google.colab import drive; drive.mount("/content/drive")
        !unzip -q -o "$ZIP_IN_DRIVE" -d /content/dataset
    else:
        from google.colab import files
        print("Tải lên kaggle.json (Kaggle -> Settings -> Create New Token)"); files.upload()
        !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/dataset --unzip
    # Tìm thư mục chứa 38-39 thư mục lớp (tên zip/cấu trúc có thể khác nhau)
    cands = [d for d in glob.glob("/content/dataset/**/", recursive=True) if len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))]) >= 38]
    assert cands, "Không tìm thấy thư mục lớp; kiểm tra lại file zip"
    DATA_DIR = sorted(cands, key=len)[0]
classes = sorted(x for x in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, x)))
print(DATA_DIR, "|", len(classes), "lớp |", sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes), "ảnh")

In [ ]:
#@title 4. Cấu hình và huấn luyện cả ba kiến trúc
EPOCHS = 8  #@param {type:"integer"}
FINE_TUNE_EPOCHS = 4  #@param {type:"integer"}
BATCH = 64  #@param {type:"integer"}
LIMIT_PER_CLASS = 0  #@param {type:"integer"}
#@markdown `LIMIT_PER_CLASS` = 0 dùng toàn bộ ảnh; đặt 100 để thử nhanh (~10 phút). Chọn kiến trúc:
RUN_MOBILENETV3 = True  #@param {type:"boolean"}
RUN_EFFICIENTNETB0 = True  #@param {type:"boolean"}
RUN_VIT = True  #@param {type:"boolean"}
archs = [a for a, on in [("mobilenetv3", RUN_MOBILENETV3), ("efficientnetb0", RUN_EFFICIENTNETB0), ("vit", RUN_VIT)] if on]
arch_args = " ".join(f"--arch {a}" for a in archs)
%cd /content
!python compare_models.py --data "$DATA_DIR" $arch_args --epochs $EPOCHS --fine_tune_epochs $FINE_TUNE_EPOCHS --batch $BATCH --limit_per_class $LIMIT_PER_CLASS --out results/compare --verbose 2

In [ ]:
#@title 5. Bảng và biểu đồ so sánh
import pandas as pd
from IPython.display import Image, display
df = pd.read_csv("results/compare/summary.csv")
cols = {"name": "Kiến trúc", "accuracy": "Accuracy", "top3": "Top-3", "macro_f1": "Macro-F1", "params_m": "Tham số (triệu)", "size_mb": "Dung lượng (MB)",
        "cpu_ms": "CPU ms/ảnh", "train_min": "Huấn luyện (phút)", "epochs": "Epoch", "best_val_acc": "Val acc tốt nhất"}
display(df[list(cols)].rename(columns=cols).style.format({"Accuracy": "{:.4f}", "Top-3": "{:.4f}", "Macro-F1": "{:.4f}", "Val acc tốt nhất": "{:.4f}"}).hide(axis="index"))
display(Image("results/compare/summary.png"))
for a in df["arch"]:
    h = pd.read_csv(f"results/compare/{a}/training_history.csv")
    print(f"{a}: val_accuracy theo epoch =", [round(v, 4) for v in h["val_accuracy"]])

In [ ]:
#@title 6. Lưu kết quả: nén results/compare và tải về (hoặc chép lên Drive)
SAVE_TO_DRIVE = ""  #@param {type:"string"}
!cd results && zip -q -r /content/compare_results.zip compare -x "*.keras"
if SAVE_TO_DRIVE:  # VD: /content/drive/MyDrive/phathiensaubenh
    from google.colab import drive; drive.mount("/content/drive")
    !mkdir -p "$SAVE_TO_DRIVE" && cp /content/compare_results.zip "$SAVE_TO_DRIVE/"
    print("Đã chép lên Drive:", SAVE_TO_DRIVE)
else:
    from google.colab import files; files.download("/content/compare_results.zip")

## Sau khi chạy xong
1. Giải nén `compare_results.zip` vào thư mục dự án để có `D:\phathiensaubenh\results\compare\summary.csv` (và các thư mục `mobilenetv3/`, `efficientnetb0/`, `vit/`).
2. Mở trang quản trị → **Đánh giá mô hình** → tab **So sánh kiến trúc**: bảng, biểu đồ và ma trận nhầm lẫn từng kiến trúc.
3. Trong báo cáo nên nêu: cùng dữ liệu/chia tập/quy trình; ViT-B/16 lớn gấp ~20 lần MobileNetV3Large (86M so với 4,2M tham số) nên dù có thể chính xác hơn vài phần trăm vẫn khó chạy trên máy yếu; EfficientNet-B0 là mức trung gian. Lý do chọn MobileNetV3Large cho ứng dụng: nhẹ, suy luận CPU nhanh, đủ chính xác trên PlantVillage.

Muốn giữ file mô hình của từng kiến trúc (để thử triển khai), thêm `--keep_models` vào lệnh ở ô 4 (file ViT ~ 330 MB).